<a href="https://colab.research.google.com/github/elattiass/Sm-Part-1/blob/main/simulationassignement1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl">

# מטלה: מערכת ניהול חדר כושר

## קבוצה: 16

### מגישי המטלה:

1. **שם:** אלעד אטיאס | **ת.ז:** 207897489
2. **שם:** איל רוטשטיין | **ת.ז:** 209142918
3. **שם:** יעל אלעזרי | **ת.ז:** 322713447

---

מחברת זו מכילה את הפתרון שלנו למטלת מערכת ניהול חדר כושר, המבוסס על תכנות מונחה עצמים.
ארגנו את הקוד בסדר פשוט: תחילה חריגות מותאמות אישית, לאחר מכן מחלקות הליבה, הרחבת השירותים הנוספים, המחלקה הראשית המאגדת הכל, ולבסוף הדגמה המציגה את המערכת בפעולה.

</div>

<div dir="rtl">

# README

מטרת המחברת היא למדל מערכת ניהול חדר כושר קטנה תוך שימוש בעקרונות תכנות מונחה עצמים, אימות קלט, טיפול בחריגות והדגמה ברורה של התכונות העיקריות.

ניסינו לשמור על עיצוב פרקטי. במקום ליצור מחלקה אחת ענקית שעושה הכל, הפרדנו אחריות בין המחלקות השונות. זה הפך את הלוגיקה לקלה יותר לבדיקה וקרובה יותר לאופן שבו מערכת כזו פועלת במציאות.

הנחות יסוד במודל:
- לכל מנוי יש סוג חברות אחד ורמת כושר נוכחית אחת.
- לכל שיעור כושר מוקצה מאמן אחד בכל עת.
- קונפליקטים בלוח הזמנים נבדקים על פי חפיפה ביום ובשעה.
- דוחות תשלום מטופלים כסיכום חיוב פשוט המבוסס על הפעילות המתועדת.
- ההרחבה היצירתית בגרסה זו היא שירות התאוששות ומוביליטי, בו ניתן להקצות למנויים תוכניות התאוששות ולקבוע פגישות עם מאמנים.

החלק האחרון של המחברת כולל הדגמה. השתמשנו בו כדי להראות זרימת עבודה רגילה, וגם כדי להוכיח שהמערכת מטפלת נכון בשגיאות, כגון מזהים כפולים, חוסר התאמה ברמה, קונפליקטים בלוח הזמנים וחוסר זמינות של מאמנים.

</div>

<div dir="rtl">

# חריגות

התחלנו בהגדרת חריגות מותאמות אישית ולניפוי שגיאות.
במקום לקבל שגיאות מערכת כלליות, המערכת יכולה להסביר בדיוק מה השתבש: קלט לא תקין, רשומות כפולות, בעיות בלוח זמנים, נתונים חסרים או בעיות בהרשמה.

זה גם עזר בזמן הבדיקות, כיוון שניתן לתפוס כל הפרת כלל עסקי בנפרד בהדגמה.

</div>

In [2]:
# This cell defines the exception hierarchy for the whole project.
# I separated the errors so each business rule can fail in a clear and readable way.

class GymError(Exception):
    """Base error for the gym management system."""


class ValidationError(GymError):
    """Raised when input data is invalid."""


class DuplicateIDError(GymError):
    """Raised when an entity ID already exists."""


class EnrollmentError(GymError):
    """Raised when a member cannot be enrolled."""


class ScheduleConflictError(GymError):
    """Raised when a schedule overlaps with an existing booking."""


class TrainerUnavailableError(GymError):
    """Raised when a trainer cannot take a class or session."""


class NotFoundError(GymError):
    """Raised when an expected entity cannot be found."""

<div dir="rtl">

# ליבה

חלק זה מכיל את מודל הנתונים הראשי של המערכת.
המחלקות מחוברות זו לזו בצורה טבעית: מנויים נרשמים לשיעורים, מאמנים מוקצים לשיעורים, ואובייקטים של לוח זמנים נמצאים בשימוש חוזר בכל מקום שבו נדרשת בדיקת התנגשות זמנים.

הוספנו גם אימות נתונים קרוב לאובייקטים עצמם. בדרך זו, נתונים שגויים נעצרים מוקדם ככל האפשר.

</div>

In [3]:
# This is the main modeling cell.
# It contains the shared validation helpers together with the core gym entities and their main rules.

from __future__ import annotations
from dataclasses import dataclass, field
from typing import Optional

ALLOWED_LEVELS = {"Beginner": 1, "Intermediate": 2, "Advanced": 3}
ALLOWED_MEMBERSHIPS = {"Monthly", "Yearly", "Premium"}

def _validate_positive_int(value: int, field_name: str) -> None:
    if not isinstance(value, int) or value <= 0:
        raise ValidationError(f"{field_name} must be a positive integer.")

def _validate_non_empty_string(value: str, field_name: str) -> str:
    if not isinstance(value, str):
        raise ValidationError(f"{field_name} must be a string.")
    normalized = value.strip()
    if not normalized:
        raise ValidationError(f"{field_name} must be provided.")
    return normalized

def _normalize_level(level: str) -> str:
    normalized = _validate_non_empty_string(level, "fitness level").title()
    if normalized not in ALLOWED_LEVELS:
        raise ValidationError(
            f"fitness level must be one of: {', '.join(ALLOWED_LEVELS)}."
        )
    return normalized

def _normalize_membership(membership_type: str) -> str:
    normalized = _validate_non_empty_string(membership_type, "membership type").title()
    if normalized not in ALLOWED_MEMBERSHIPS:
        raise ValidationError(
            f"membership type must be one of: {', '.join(ALLOWED_MEMBERSHIPS)}."
        )
    return normalized


@dataclass(frozen=True)
class ScheduleSlot:
    day_of_week: str
    start_time: str
    duration: int

    def __post_init__(self) -> None:
        _validate_positive_int(self.duration, "duration")
        object.__setattr__(self, "day_of_week", _validate_non_empty_string(self.day_of_week, "day_of_week"))
        object.__setattr__(self, "start_time", _validate_non_empty_string(self.start_time, "start_time"))
        self._start_minutes()

    def _start_minutes(self) -> int:
        start_time = _validate_non_empty_string(self.start_time, "start_time")
        parts = start_time.split(":")
        if len(parts) != 2:
            raise ValidationError("start_time must use HH:MM format.")
        try:
            hour = int(parts[0])
            minute = int(parts[1])
        except ValueError as exc:
            raise ValidationError("start_time must use numeric HH:MM values.") from exc
        if not (0 <= hour <= 23 and 0 <= minute <= 59):
            raise ValidationError("start_time must be a valid 24-hour time.")
        return hour * 60 + minute

    @property
    def end_minutes(self) -> int:
        return self._start_minutes() + self.duration

    def conflicts_with(self, other: Optional["ScheduleSlot"]) -> bool:
        if other is None:
            return False
        if self.day_of_week.strip().lower() != other.day_of_week.strip().lower():
            return False
        return not (
            self.end_minutes <= other._start_minutes()
            or other.end_minutes <= self._start_minutes()
        )

    def __str__(self) -> str:
        return f"{self.day_of_week} {self.start_time} ({self.duration} min)"


@dataclass
class PersonalTrainingSession:
    member_id: str
    trainer_id: str
    slot: ScheduleSlot
    fee: float = 45.0
    status: str = "Scheduled"


@dataclass
class RecoverySession:
    session_id: str
    member_id: str
    trainer_id: str
    slot: ScheduleSlot
    session_type: str
    fee: float
    status: str = "Scheduled"
    notes: str = ""

    def __post_init__(self) -> None:
        self.session_id = _validate_non_empty_string(self.session_id, "session_id")
        self.member_id = _validate_non_empty_string(self.member_id, "member_id")
        self.trainer_id = _validate_non_empty_string(self.trainer_id, "trainer_id")
        self.session_type = _validate_non_empty_string(self.session_type, "session_type")
        if not isinstance(self.slot, ScheduleSlot):
            raise ValidationError("slot must be a ScheduleSlot instance.")
        if not isinstance(self.fee, (int, float)) or self.fee < 0:
            raise ValidationError("fee must be a non-negative number.")
        self.status = _validate_non_empty_string(self.status, "status").title()
        if not isinstance(self.notes, str):
            raise ValidationError("notes must be a string.")
        self.notes = self.notes.strip()


@dataclass
class Member:
    member_id: str
    name: str
    age: int
    membership_type: str
    fitness_level: str
    current_enrollments: list["FitnessClass"] = field(default_factory=list)
    completed_classes: list[str] = field(default_factory=list)
    cancellations_count: int = 0
    no_show_count: int = 0
    personal_training_sessions: list[PersonalTrainingSession] = field(default_factory=list)
    recovery_plan_ids: list[str] = field(default_factory=list)
    recovery_monthly_fees: list[float] = field(default_factory=list)
    recovery_sessions: list[RecoverySession] = field(default_factory=list)

    def __post_init__(self) -> None:
        _validate_positive_int(self.age, "age")
        self.member_id = _validate_non_empty_string(self.member_id, "member_id")
        self.name = _validate_non_empty_string(self.name, "member name")
        self.membership_type = _normalize_membership(self.membership_type)
        self.fitness_level = _normalize_level(self.fitness_level)

    def enroll_to_class(self, fitness_class: "FitnessClass") -> str:
        return fitness_class.enroll(self)

    def cancel_enrollment(self, class_id: str) -> str:
        for fitness_class in self.current_enrollments:
            if fitness_class.class_id == class_id:
                message = fitness_class.drop(self.member_id)
                self.cancellations_count += 1
                return f"{self.name} cancelled {fitness_class.name}. {message}"
        raise NotFoundError(
            f"Member {self.name} is not currently enrolled in class {class_id}."
        )

    def add_completed_class(self, class_id: str) -> None:
        if class_id not in self.completed_classes:
            self.completed_classes.append(class_id)

    def request_personal_training(
        self, trainer: "Trainer", time_slot: ScheduleSlot
    ) -> PersonalTrainingSession:
        if not self.can_attend(time_slot):
            raise ScheduleConflictError(
                f"{self.name} already has another activity during {time_slot}."
            )
        return trainer.book_personal_training(self, time_slot)

    def request_recovery_session(
        self,
        service: "RecoveryService",
        trainer: "Trainer",
        session_id: str,
        time_slot: ScheduleSlot,
        session_type: str,
        notes: str = "",
    ) -> RecoverySession:
        if not self.can_attend(time_slot):
            raise ScheduleConflictError(
                f"{self.name} already has another activity during {time_slot}."
            )
        return service.book_session(session_id, self, trainer, time_slot, session_type, notes)

    def can_attend(self, slot: ScheduleSlot) -> bool:
        for fitness_class in self.current_enrollments:
            if fitness_class.schedule_slot and fitness_class.schedule_slot.conflicts_with(slot):
                return False
        for session in self.personal_training_sessions:
            if session.status == "Scheduled" and session.slot.conflicts_with(slot):
                return False
        for session in self.recovery_sessions:
            if session.status == "Scheduled" and session.slot.conflicts_with(slot):
                return False
        return True

    def record_no_show(self) -> None:
        self.no_show_count += 1

    def generate_payment_report(self) -> dict[str, float | int | str]:
        pricing = {
            "Monthly": {"base_fee": 75.0, "included_classes": 8},
            "Yearly": {"base_fee": 55.0, "included_classes": 10},
            "Premium": {"base_fee": 110.0, "included_classes": float("inf")},
        }
        extra_class_fee = 8.0
        cancellation_penalty = 5.0
        no_show_penalty = 12.0

        plan = pricing[self.membership_type]
        completed_count = len(self.completed_classes)
        included_classes = plan["included_classes"]
        extra_classes = (
            0
            if included_classes == float("inf")
            else max(0, completed_count - int(included_classes))
        )
        class_charges = extra_classes * extra_class_fee
        cancellation_charges = self.cancellations_count * cancellation_penalty
        no_show_charges = self.no_show_count * no_show_penalty
        personal_training_charges = sum(
            session.fee for session in self.personal_training_sessions if session.status == "Scheduled"
        )
        recovery_plan_charges = sum(self.recovery_monthly_fees)
        recovery_session_charges = sum(
            session.fee for session in self.recovery_sessions if session.status == "Scheduled"
        )
        total = (
            plan["base_fee"]
            + class_charges
            + cancellation_charges
            + no_show_charges
            + personal_training_charges
            + recovery_plan_charges
            + recovery_session_charges
        )
        return {
            "member_id": self.member_id,
            "member_name": self.name,
            "membership_type": self.membership_type,
            "base_fee": plan["base_fee"],
            "completed_classes": completed_count,
            "extra_class_charges": class_charges,
            "cancellation_penalties": cancellation_charges,
            "no_show_penalties": no_show_charges,
            "personal_training_charges": personal_training_charges,
            "recovery_plan_charges": recovery_plan_charges,
            "recovery_session_charges": recovery_session_charges,
            "total_due": total,
        }


@dataclass
class Trainer:
    trainer_id: str
    name: str
    specialties: list[str]
    experience_years: int
    max_weekly_classes: int
    assigned_classes: list["FitnessClass"] = field(default_factory=list)
    personal_training_sessions: list[PersonalTrainingSession] = field(default_factory=list)
    recovery_sessions: list[RecoverySession] = field(default_factory=list)

    def __post_init__(self) -> None:
        _validate_positive_int(self.max_weekly_classes, "max_weekly_classes")
        if not isinstance(self.experience_years, int) or self.experience_years < 0:
            raise ValidationError("experience_years must be zero or a positive integer.")
        self.trainer_id = _validate_non_empty_string(self.trainer_id, "trainer_id")
        self.name = _validate_non_empty_string(self.name, "trainer name")
        if not isinstance(self.specialties, list) or not self.specialties:
            raise ValidationError("specialties must be a non-empty list.")
        self.specialties = [
            _validate_non_empty_string(specialty, "specialty")
            for specialty in self.specialties
        ]

    def assign_class(self, fitness_class: "FitnessClass") -> str:
        if fitness_class in self.assigned_classes:
            raise TrainerUnavailableError(
                f"{self.name} is already assigned to class {fitness_class.class_id}."
            )
        if len(self.assigned_classes) >= self.max_weekly_classes:
            raise TrainerUnavailableError(
                f"{self.name} cannot exceed {self.max_weekly_classes} weekly classes."
            )
        if fitness_class.schedule_slot and not self.is_available(fitness_class.schedule_slot):
            raise TrainerUnavailableError(
                f"{self.name} is not available for {fitness_class.schedule_slot}."
            )
        if fitness_class.trainer and fitness_class.trainer is not self:
            fitness_class.trainer.remove_class(fitness_class.class_id)
        self.assigned_classes.append(fitness_class)
        fitness_class.trainer = self
        return f"{self.name} assigned to {fitness_class.name}."

    def remove_class(self, class_id: str) -> str:
        for fitness_class in self.assigned_classes:
            if fitness_class.class_id == class_id:
                self.assigned_classes.remove(fitness_class)
                if fitness_class.trainer is self:
                    fitness_class.trainer = None
                return f"{self.name} removed from {fitness_class.name}."
        raise NotFoundError(f"Trainer {self.name} is not assigned to class {class_id}.")

    def is_available(self, slot: ScheduleSlot) -> bool:
        for fitness_class in self.assigned_classes:
            if fitness_class.schedule_slot and fitness_class.schedule_slot.conflicts_with(slot):
                return False
        for session in self.personal_training_sessions:
            if session.status == "Scheduled" and session.slot.conflicts_with(slot):
                return False
        for session in self.recovery_sessions:
            if session.status == "Scheduled" and session.slot.conflicts_with(slot):
                return False
        return True

    def analyze_training_statistics(self) -> dict[str, object]:
        category_breakdown: dict[str, int] = {}
        total_members = 0
        for fitness_class in self.assigned_classes:
            category_breakdown[fitness_class.category] = (
                category_breakdown.get(fitness_class.category, 0) + 1
            )
            total_members += len(fitness_class.enrolled_members)
        return {
            "trainer_name": self.name,
            "group_classes": len(self.assigned_classes),
            "personal_training_sessions": len(self.personal_training_sessions),
            "recovery_sessions": len(self.recovery_sessions),
            "members_reached": total_members,
            "category_breakdown": category_breakdown,
        }

    def book_personal_training(
        self, member: Member, slot: ScheduleSlot
    ) -> PersonalTrainingSession:
        if not self.is_available(slot):
            raise TrainerUnavailableError(
                f"{self.name} is already booked during {slot}."
            )
        session = PersonalTrainingSession(
            member_id=member.member_id,
            trainer_id=self.trainer_id,
            slot=slot,
            fee=45.0 + (self.experience_years * 2.0),
        )
        self.personal_training_sessions.append(session)
        member.personal_training_sessions.append(session)
        return session


@dataclass
class FitnessClass:
    class_id: str
    name: str
    category: str
    difficulty_level: str
    capacity: int
    duration: int
    trainer: Optional[Trainer] = None
    schedule_slot: Optional[ScheduleSlot] = None
    enrolled_members: list[Member] = field(default_factory=list)
    waiting_list: list[Member] = field(default_factory=list)

    def __post_init__(self) -> None:
        _validate_positive_int(self.capacity, "capacity")
        _validate_positive_int(self.duration, "duration")
        self.class_id = _validate_non_empty_string(self.class_id, "class_id")
        self.name = _validate_non_empty_string(self.name, "class name")
        self.category = _validate_non_empty_string(self.category, "category")
        self.difficulty_level = _normalize_level(self.difficulty_level)

    def assign_trainer(self, trainer: Trainer) -> str:
        return trainer.assign_class(self)

    def set_schedule(self, schedule_slot: ScheduleSlot) -> str:
        if self.trainer:
            for assigned_class in self.trainer.assigned_classes:
                if assigned_class is self:
                    continue
                if assigned_class.schedule_slot and assigned_class.schedule_slot.conflicts_with(
                    schedule_slot
                ):
                    raise TrainerUnavailableError(
                        f"{self.trainer.name} has a conflict with {assigned_class.name}."
                    )
        self.schedule_slot = schedule_slot
        return f"{self.name} scheduled for {schedule_slot}."

    def has_capacity(self) -> bool:
        return len(self.enrolled_members) < self.capacity

    def is_suitable(self, member: Member) -> bool:
        return ALLOWED_LEVELS[member.fitness_level] >= ALLOWED_LEVELS[self.difficulty_level]

    def enroll(self, member: Member) -> str:
        if self.schedule_slot is None:
            raise EnrollmentError(f"{self.name} cannot accept enrollments before scheduling.")
        if member in self.enrolled_members:
            raise EnrollmentError(f"{member.name} is already enrolled in {self.name}.")
        if member in self.waiting_list:
            raise EnrollmentError(f"{member.name} is already on the waiting list for {self.name}.")
        if not self.is_suitable(member):
            raise EnrollmentError(
                f"{member.name} is {member.fitness_level} and cannot join {self.difficulty_level} {self.name}."
            )
        if not member.can_attend(self.schedule_slot):
            raise ScheduleConflictError(
                f"{member.name} has a schedule conflict and cannot attend {self.name}."
            )

        if self.has_capacity():
            self.enrolled_members.append(member)
            member.current_enrollments.append(self)
            return f"{member.name} enrolled successfully in {self.name}."

        self.add_to_waiting_list(member)
        return f"{self.name} is full. {member.name} was added to the waiting list."

    def drop(self, member_id: str) -> str:
        for member in self.enrolled_members:
            if member.member_id == member_id:
                self.enrolled_members.remove(member)
                if self in member.current_enrollments:
                    member.current_enrollments.remove(self)
                promotion_message = self.promote_from_waiting_list()
                if promotion_message:
                    return (
                        f"{member.name} dropped from {self.name}. {promotion_message}"
                    )
                return f"{member.name} dropped from {self.name}."
        for member in self.waiting_list:
            if member.member_id == member_id:
                self.waiting_list.remove(member)
                return f"{member.name} removed from the waiting list for {self.name}."
        raise NotFoundError(f"Member {member_id} was not found in {self.name}.")

    def add_to_waiting_list(self, member: Member) -> None:
        if member not in self.waiting_list:
            self.waiting_list.append(member)

    def promote_from_waiting_list(self) -> str:
        if not self.waiting_list or not self.has_capacity() or self.schedule_slot is None:
            return ""

        candidate = self.waiting_list[0]
        if not self.is_suitable(candidate):
            return f"{candidate.name} remains first on the waiting list because the class level is not suitable."
        if not candidate.can_attend(self.schedule_slot):
            return f"{candidate.name} remains first on the waiting list due to a schedule conflict."

        promoted = self.waiting_list.pop(0)
        self.enrolled_members.append(promoted)
        promoted.current_enrollments.append(self)
        return f"{promoted.name} was promoted from the waiting list."


<div dir="rtl">

# הרחבות

המטלה ביקשה הרחבה יצירתית, ולכן בחרנו בשירות התאוששות ומוביליטי.
בחרנו ברעיון הזה כי הוא מתאים לתחום חדר הכושר ומתחבר בצורה טבעית ללוחות זמנים, מאמנים ותשלומים.

בחלק זה, כל תוכנית התאוששות שומרת את הפרטים שלה, ושירות ההתאוששות מנהל את התוכניות הזמינות ואת הפגישות שהוזמנו.

</div>

In [4]:
# This cell contains the extra feature I added beyond the basic gym requirements.
# The recovery service extends the system with another type of member support, scheduling, and billing.

from __future__ import annotations
from dataclasses import dataclass, field

@dataclass
class RecoveryPlan:
    plan_id: str
    title: str
    focus_area: str
    duration_weeks: int
    monthly_fee: float
    assigned_members: list[str] = field(default_factory=list)

    def __post_init__(self) -> None:
        self.plan_id = _validate_non_empty_string(self.plan_id, "plan_id")
        self.title = _validate_non_empty_string(self.title, "title")
        self.focus_area = _validate_non_empty_string(self.focus_area, "focus_area")
        _validate_positive_int(self.duration_weeks, "duration_weeks")
        if not isinstance(self.monthly_fee, (int, float)) or self.monthly_fee < 0:
            raise ValidationError("monthly_fee must be a non-negative number.")

    def assign_to_member(self, member: Member) -> str:
        if member.member_id not in self.assigned_members:
            self.assigned_members.append(member.member_id)
        if self.plan_id not in member.recovery_plan_ids:
            member.recovery_plan_ids.append(self.plan_id)
            member.recovery_monthly_fees.append(float(self.monthly_fee))
        return f"{self.title} assigned to {member.name}."

    def summary(self) -> str:
        return (
            f"{self.title} ({self.focus_area}) - {self.duration_weeks} weeks, "
            f"${self.monthly_fee:.2f}/month"
        )


@dataclass
class RecoveryService:
    service_id: str
    specialist_name: str
    plans: dict[str, RecoveryPlan] = field(default_factory=dict)
    sessions: dict[str, RecoverySession] = field(default_factory=dict)

    def __post_init__(self) -> None:
        self.service_id = _validate_non_empty_string(self.service_id, "service_id")
        self.specialist_name = _validate_non_empty_string(self.specialist_name, "specialist_name")

    def add_plan(self, plan: RecoveryPlan) -> str:
        if plan.plan_id in self.plans:
            raise DuplicateIDError(f"Recovery plan ID {plan.plan_id} already exists.")
        self.plans[plan.plan_id] = plan
        return f"Recovery plan {plan.title} added under specialist {self.specialist_name}."

    def assign_plan(self, plan_id: str, member: Member) -> str:
        if plan_id not in self.plans:
            raise NotFoundError(f"Recovery plan {plan_id} was not found.")
        return self.plans[plan_id].assign_to_member(member)

    def book_session(
        self,
        session_id: str,
        member: Member,
        trainer: Trainer,
        slot: ScheduleSlot,
        session_type: str,
        notes: str = "",
    ) -> RecoverySession:
        session_id = _validate_non_empty_string(session_id, "session_id")
        if session_id in self.sessions:
            raise DuplicateIDError(f"Recovery session ID {session_id} already exists.")
        if not member.can_attend(slot):
            raise ScheduleConflictError(
                f"{member.name} already has another activity during {slot}."
            )
        if not trainer.is_available(slot):
            raise TrainerUnavailableError(
                f"{trainer.name} is already booked during {slot}."
            )

        session = RecoverySession(
            session_id=session_id,
            member_id=member.member_id,
            trainer_id=trainer.trainer_id,
            slot=slot,
            session_type=session_type,
            fee=30.0 + (trainer.experience_years * 1.5),
            notes=notes,
        )
        self.sessions[session.session_id] = session
        member.recovery_sessions.append(session)
        trainer.recovery_sessions.append(session)
        return session

    def get_member_plans(self, member_id: str) -> list[RecoveryPlan]:
        return [
            plan for plan in self.plans.values() if member_id in plan.assigned_members
        ]

    def generate_service_report(self) -> str:
        lines = [f"Recovery Service by {self.specialist_name}"]
        if not self.plans:
            lines.append("- No recovery plans configured.")
        else:
            for plan in self.plans.values():
                lines.append(
                    f"- Plan: {plan.summary()} | Assigned members: {len(plan.assigned_members)}"
                )
        if not self.sessions:
            lines.append("- No recovery sessions booked.")
        else:
            for session in self.sessions.values():
                lines.append(
                    f"- Session {session.session_id}: Member {session.member_id} with Trainer {session.trainer_id} | "
                    f"{session.session_type} at {session.slot} | Fee: ${session.fee:.2f} | Status: {session.status}"
                )
        return "\n".join(lines)


<div dir="rtl">

# חדר הכושר

תא זה מחבר את הכל יחד.
המחלקה של חדר הכושר פועלת כמנהלת הראשית של המערכת: היא מאחסנת את המנויים, המאמנים, השיעורים ושירות ההתאוששות, והיא גם אחראית על הפקת דוחות סיכום.

שמרנו על המחלקה הזו ממוקדת בתיאום במקום להכניס לתוכה את כל הכללים. רוב האימותים נשארים באובייקט הרלוונטי, בעוד שחדר הכושר מטפל בפעולות ברמת האוסף, כגון הוספת רשומות, חיפוש והפקת דוחות.

</div>

In [5]:
# This cell defines the main container object for the whole system.
# The Gym class connects the different entities and produces the higher-level reports.

from __future__ import annotations
from dataclasses import dataclass, field

@dataclass
class Gym:
    name: str
    members: dict[str, Member] = field(default_factory=dict)
    trainers: dict[str, Trainer] = field(default_factory=dict)
    classes: dict[str, FitnessClass] = field(default_factory=dict)
    recovery_services: dict[str, RecoveryService] = field(default_factory=dict)

    def add_member(self, member: Member) -> str:
        if member.member_id in self.members:
            raise DuplicateIDError(f"Member ID {member.member_id} already exists.")
        self.members[member.member_id] = member
        return f"Member {member.name} added to {self.name}."

    def add_trainer(self, trainer: Trainer) -> str:
        if trainer.trainer_id in self.trainers:
            raise DuplicateIDError(f"Trainer ID {trainer.trainer_id} already exists.")
        self.trainers[trainer.trainer_id] = trainer
        return f"Trainer {trainer.name} added to {self.name}."

    def add_class(self, fitness_class: FitnessClass) -> str:
        if fitness_class.class_id in self.classes:
            raise DuplicateIDError(f"Class ID {fitness_class.class_id} already exists.")
        self.classes[fitness_class.class_id] = fitness_class
        return f"Class {fitness_class.name} added to {self.name}."

    def add_recovery_service(self, service: RecoveryService) -> str:
        if service.service_id in self.recovery_services:
            raise DuplicateIDError(f"Recovery service ID {service.service_id} already exists.")
        self.recovery_services[service.service_id] = service
        return f"Recovery service by {service.specialist_name} added to {self.name}."

    def generate_schedule_report(self) -> str:
        def sort_key(fitness_class: FitnessClass) -> tuple[str, str]:
            if fitness_class.schedule_slot:
                return (
                    fitness_class.schedule_slot.day_of_week.lower(),
                    fitness_class.schedule_slot.start_time,
                )
            return ("zzzz", "99:99")

        lines = [f"Schedule Report for {self.name}"]
        overlaps: list[str] = []
        ordered_classes = sorted(self.classes.values(), key=sort_key)

        for fitness_class in ordered_classes:
            trainer_name = fitness_class.trainer.name if fitness_class.trainer else "Unassigned"
            schedule = str(fitness_class.schedule_slot) if fitness_class.schedule_slot else "Not scheduled"
            lines.append(
                f"- {fitness_class.class_id}: {fitness_class.name} | {fitness_class.category} | "
                f"{fitness_class.difficulty_level} | Trainer: {trainer_name} | "
                f"Schedule: {schedule} | Enrolled: {len(fitness_class.enrolled_members)}/{fitness_class.capacity}"
            )

        classes_list = list(self.classes.values())
        for index, current_class in enumerate(classes_list):
            for other_class in classes_list[index + 1 :]:
                if (
                    current_class.schedule_slot
                    and other_class.schedule_slot
                    and current_class.schedule_slot.conflicts_with(other_class.schedule_slot)
                ):
                    overlaps.append(
                        f"Overlap detected: {current_class.name} conflicts with {other_class.name}."
                    )

        if overlaps:
            lines.append("Warnings:")
            lines.extend(f"- {warning}" for warning in overlaps)
        else:
            lines.append("No schedule overlaps found.")

        return "\n".join(lines)

    def search_classes(self, term: str) -> list[FitnessClass]:
        if not isinstance(term, str):
            raise ValidationError("search term must be a string.")
        lookup = term.strip().lower()
        if not lookup:
            return list(self.classes.values())
        return [
            fitness_class
            for fitness_class in self.classes.values()
            if lookup in fitness_class.name.lower()
            or lookup in fitness_class.category.lower()
            or lookup in fitness_class.difficulty_level.lower()
            or (
                fitness_class.trainer is not None
                and lookup in fitness_class.trainer.name.lower()
            )
        ]

    def generate_membership_report(self) -> str:
        summary: dict[str, int] = {}
        for member in self.members.values():
            summary[member.membership_type] = summary.get(member.membership_type, 0) + 1

        lines = [f"Membership Report for {self.name}"]
        for membership_type in sorted(summary):
            lines.append(f"- {membership_type}: {summary[membership_type]} active members")
        lines.append(f"- Total active members: {len(self.members)}")
        return "\n".join(lines)

    def generate_recovery_report(self) -> str:
        lines = [f"Recovery Services Report for {self.name}"]
        if not self.recovery_services:
            lines.append("- No recovery services configured.")
            return "\n".join(lines)
        for service in self.recovery_services.values():
            lines.append(service.generate_service_report())
        return "\n".join(lines)


<div dir="rtl">

# MAIN

חלק אחרון זה הוא חלק ההדגמה של המטלה.
השתמשנו בו כדי להראות את המערכת בפעולה מהתחלה ועד הסוף: יצירת חדר הכושר, הוספת אנשים ושיעורים, בדיקת הרשמה, בדיקת מקרי קצה, שימוש בכלל השירותים והדפסת דוחות.


</div>

In [6]:
# This cell contains the helper print functions and the full demo scenario.
# Running it should produce a readable walk-through of the system, including normal cases and expected failures.

from __future__ import annotations

def print_section(title: str) -> None:
    print(f"\n{'=' * 18} {title} {'=' * 18}")

def print_payment_report(member: Member) -> None:
    report = member.generate_payment_report()
    print(f"Payment report for {report['member_name']}:")
    print(f"- Membership: {report['membership_type']}")
    print(f"- Base fee: ${report['base_fee']:.2f}")
    print(f"- Completed classes: {report['completed_classes']}")
    print(f"- Extra class charges: ${report['extra_class_charges']:.2f}")
    print(f"- Cancellation penalties: ${report['cancellation_penalties']:.2f}")
    print(f"- No-show penalties: ${report['no_show_penalties']:.2f}")
    print(f"- Personal training charges: ${report['personal_training_charges']:.2f}")
    print(f"- Recovery plan charges: ${report['recovery_plan_charges']:.2f}")
    print(f"- Recovery session charges: ${report['recovery_session_charges']:.2f}")
    print(f"- Total due: ${report['total_due']:.2f}")

def print_trainer_stats(trainer: Trainer) -> None:
    stats = trainer.analyze_training_statistics()
    print(f"Trainer statistics for {stats['trainer_name']}:")
    print(f"- Group classes: {stats['group_classes']}")
    print(f"- Personal training sessions: {stats['personal_training_sessions']}")
    print(f"- Recovery sessions: {stats['recovery_sessions']}")
    print(f"- Members reached: {stats['members_reached']}")
    print(f"- Category breakdown: {stats['category_breakdown']}")

def run_demo() -> None:
    print_section("Create Gym")
    gym = Gym("Pulse Point Gym")
    print(f"Gym created: {gym.name}")

    print_section("Create Trainers")
    alice = Trainer("T100", "Alice Kim", ["Yoga", "Pilates"], experience_years=6, max_weekly_classes=3)
    ben = Trainer("T200", "Ben Ortiz", ["HIIT", "Cycling"], experience_years=8, max_weekly_classes=2)
    cara = Trainer("T300", "Cara Stone", ["Strength", "Mobility"], experience_years=10, max_weekly_classes=2)
    for trainer in (alice, ben, cara):
        print(gym.add_trainer(trainer))

    print_section("Create Members")
    mia = Member("M100", "Mia Chen", 25, "Monthly", "Beginner")
    noah = Member("M200", "Noah Patel", 31, "Yearly", "Intermediate")
    liam = Member("M300", "Liam Brooks", 29, "Premium", "Advanced")
    ava = Member("M400", "Ava Rivera", 24, "Yearly", "Beginner")
    zoe = Member("M500", "Zoe Adams", 27, "Monthly", "Beginner")
    for member in (mia, noah, liam, ava, zoe):
        print(gym.add_member(member))

    print_section("Create Classes")
    yoga = FitnessClass("C100", "Morning Yoga", "Mind-Body", "Beginner", capacity=2, duration=60)
    hiit = FitnessClass("C200", "HIIT Blast", "Cardio", "Intermediate", capacity=2, duration=45)
    power_lift = FitnessClass("C300", "Power Lift", "Strength", "Advanced", capacity=1, duration=60)
    core_balance = FitnessClass("C400", "Core Balance", "Pilates", "Beginner", capacity=3, duration=45)
    conflict_class = FitnessClass("C500", "Stretch Express", "Recovery", "Beginner", capacity=4, duration=30)

    for fitness_class in (yoga, hiit, power_lift, core_balance):
        print(gym.add_class(fitness_class))

    print(yoga.set_schedule(ScheduleSlot("Monday", "09:00", 60)))
    print(hiit.set_schedule(ScheduleSlot("Monday", "09:30", 45)))
    print(power_lift.set_schedule(ScheduleSlot("Tuesday", "18:00", 60)))
    print(core_balance.set_schedule(ScheduleSlot("Wednesday", "17:00", 45)))
    print(conflict_class.set_schedule(ScheduleSlot("Monday", "09:15", 30)))

    print(yoga.assign_trainer(alice))
    print(hiit.assign_trainer(ben))
    print(power_lift.assign_trainer(cara))
    print(core_balance.assign_trainer(alice))

    try:
        print(conflict_class.assign_trainer(alice))
    except TrainerUnavailableError as error:
        print(f"Trainer assignment failed as expected: {error}")

    print_section("Direct Method Checks")
    print(f"Morning Yoga has capacity before enrollments? {yoga.has_capacity()}")
    print(f"Can Mia attend Morning Yoga before enrolling? {mia.can_attend(yoga.schedule_slot)}")
    print(f"Is Power Lift suitable for Mia? {power_lift.is_suitable(mia)}")
    print(f"Is Alice available for Stretch Express? {alice.is_available(conflict_class.schedule_slot)}")
    print(alice.remove_class("C400"))
    print(core_balance.assign_trainer(alice))

    print_section("Enroll Members")
    print(mia.enroll_to_class(yoga))
    print(ava.enroll_to_class(yoga))
    print(zoe.enroll_to_class(yoga))
    print(f"Morning Yoga waiting list now: {[member.name for member in yoga.waiting_list]}")

    print(noah.enroll_to_class(hiit))

    try:
        print(noah.enroll_to_class(conflict_class))
    except ScheduleConflictError as error:
        print(f"Enrollment blocked by schedule conflict: {error}")

    try:
        print(mia.enroll_to_class(power_lift))
    except EnrollmentError as error:
        print(f"Enrollment blocked by level mismatch: {error}")

    print_section("Drop And Waiting List Promotion")
    print(ava.cancel_enrollment("C100"))
    print(f"Morning Yoga enrolled members: {[member.name for member in yoga.enrolled_members]}")
    print(f"Morning Yoga waiting list: {[member.name for member in yoga.waiting_list]}")

    print_section("Edge Case Testing")

    print("Test 1: Duplicate Member ID")
    try:
        gym.add_member(Member("M100", "Duplicate Mia", 30, "Monthly", "Beginner"))
    except DuplicateIDError as error:
        print(f"Caught expected error: {error}")

    print("\nTest 2: Trainer Weekly Limit")
    test_class_1 = FitnessClass("C901", "Test Class 1", "Cardio", "Beginner", 10, 30)
    test_class_2 = FitnessClass("C902", "Test Class 2", "Cardio", "Beginner", 10, 30)
    gym.add_class(test_class_1)
    gym.add_class(test_class_2)
    test_class_1.set_schedule(ScheduleSlot("Friday", "10:00", 30))
    test_class_2.set_schedule(ScheduleSlot("Friday", "11:00", 30))

    print(test_class_1.assign_trainer(ben))
    try:
        print(test_class_2.assign_trainer(ben))
    except TrainerUnavailableError as error:
        print(f"Caught expected error: {error}")

    print("\nTest 3: Dropping Non-Enrolled Member")
    try:
        liam.cancel_enrollment("C100")
    except NotFoundError as error:
        print(f"Caught expected error: {error}")

    print("\nTest 4: Schedule Conflict on Update")
    try:
        core_balance.set_schedule(ScheduleSlot("Monday", "09:00", 45))
    except TrainerUnavailableError as error:
        print(f"Caught expected error: {error}")

    print_section("Personal Training")
    personal_training_slot = ScheduleSlot("Thursday", "07:00", 60)
    session = liam.request_personal_training(ben, personal_training_slot)
    print(
        f"Personal training booked: Member {liam.name} with Trainer {ben.name} "
        f"at {session.slot} for ${session.fee:.2f}"
    )

    print_section("Creative Extension - Recovery Service")
    recovery_service = RecoveryService("R100", "Dana Wells")
    print(gym.add_recovery_service(recovery_service))
    mobility_reset = RecoveryPlan("RP100", "Mobility Reset", "Joint mobility and flexibility", 6, 28.0)
    post_workout = RecoveryPlan("RP200", "Post-Workout Recovery", "Muscle recovery and cooldown", 4, 22.0)
    print(recovery_service.add_plan(mobility_reset))
    print(recovery_service.add_plan(post_workout))
    print(recovery_service.assign_plan("RP100", liam))
    print(recovery_service.assign_plan("RP200", mia))
    liam_plans = recovery_service.get_member_plans(liam.member_id)
    print(f"Liam's recovery plans: {[plan.title for plan in liam_plans]}")

    recovery_slot = ScheduleSlot("Friday", "07:30", 45)
    recovery_session = liam.request_recovery_session(
        recovery_service,
        cara,
        "RS100",
        recovery_slot,
        "Mobility Reset Session",
        "Focus on hips and lower back.",
    )
    print(
        f"Recovery session booked: Member {liam.name} with Trainer {cara.name} "
        f"at {recovery_session.slot} for ${recovery_session.fee:.2f}"
    )

    try:
        mia.request_recovery_session(
            recovery_service,
            ben,
            "RS200",
            ScheduleSlot("Monday", "09:15", 30),
            "Express Recovery",
            "Attempt during class time.",
        )
    except ScheduleConflictError as error:
        print(f"Recovery booking blocked by schedule conflict: {error}")

    try:
        noah.request_recovery_session(
            recovery_service,
            ben,
            "RS201",
            ScheduleSlot("Friday", "10:00", 30),
            "Cycling Recovery",
            "Attempt while trainer is teaching.",
        )
    except TrainerUnavailableError as error:
        print(f"Recovery booking blocked by trainer availability: {error}")

    print(gym.generate_recovery_report())

    print_section("Payment Reports")
    mia.add_completed_class("C100")
    mia.add_completed_class("C400")
    ava.add_completed_class("C100")
    ava.record_no_show()
    liam.add_completed_class("C300")
    liam.record_no_show()
    print_payment_report(mia)
    print_payment_report(ava)
    print_payment_report(liam)

    print_section("Search Classes")
    search_results = gym.search_classes("alice")
    print(f"Search results for 'alice': {[fitness_class.name for fitness_class in search_results]}")
    search_results = gym.search_classes("beginner")
    print(f"Search results for 'beginner': {[fitness_class.name for fitness_class in search_results]}")

    print_section("Trainer Statistics")
    print_trainer_stats(alice)
    print_trainer_stats(ben)
    print_trainer_stats(cara)

    print_section("Schedule Report")
    print(gym.generate_schedule_report())

    print_section("Membership Report")
    print(gym.generate_membership_report())


if __name__ == "__main__":
    run_demo()



================== Create Gym ==================
Gym created: Pulse Point Gym

================== Create Trainers ==================
Trainer Alice Kim added to Pulse Point Gym.
Trainer Ben Ortiz added to Pulse Point Gym.
Trainer Cara Stone added to Pulse Point Gym.

================== Create Members ==================
Member Mia Chen added to Pulse Point Gym.
Member Noah Patel added to Pulse Point Gym.
Member Liam Brooks added to Pulse Point Gym.
Member Ava Rivera added to Pulse Point Gym.
Member Zoe Adams added to Pulse Point Gym.

================== Create Classes ==================
Class Morning Yoga added to Pulse Point Gym.
Class HIIT Blast added to Pulse Point Gym.
Class Power Lift added to Pulse Point Gym.
Class Core Balance added to Pulse Point Gym.
Morning Yoga scheduled for Monday 09:00 (60 min).
HIIT Blast scheduled for Monday 09:30 (45 min).
Power Lift scheduled for Tuesday 18:00 (60 min).
Core Balance scheduled for Wednesday 17:00 (45 min).
Stretch Express scheduled for M